# מפגש 3 — השוואת baseline לרגרסיה לוגיסטית
מחברת נפרדת השומרת על הניתוח המקורי. Run All מחשב מחדש את הניסוי באמצעות ecg_session3.py.
כל הקלטה נמצאת רק באימון או בבדיקה; המזהים אינם מאפיינים. הסבר מלא ב־SESSION3_README.md. התוצרים נשמרים מקומית ב־derived_session3.
ה־baseline בונה תבניות ממוצעות מהאימון ומשווה בדמיון קוסינוס. Logistic Regression לומד מ־350 ערכי כל פעימה, אחרי StandardScaler בתוך Pipeline. ברמת הקלטה ממוצעים את הסתברויות הפעימות.

In [1]:
from pathlib import Path
import sys
from IPython.display import display
project = Path.cwd()
if not (project / 'ecg_session3.py').exists() and (project.parent / 'ecg_session3.py').exists():
    project = project.parent
if not (project / 'ecg_session3.py').exists():
    raise FileNotFoundError('Open this notebook from the ECG project root')
sys.path.insert(0, str(project))
from ecg_session3 import run
experiment = run()
display(experiment['summary'])
print({k: experiment['metadata'][k] for k in ['train_recordings', 'test_recordings', 'train_beats', 'test_beats', 'all_beats', 'classes', 'excluded_recordings', 'iterations', 'source_files_unchanged']})

,method,unit,total,correct,errors,accuracy,macro_precision,macro_recall,macro_f1-score,weighted_precision,weighted_recall,weighted_f1-score
0,baseline,recording,102,88,14,0.862745,0.859551,0.889700,0.864125,0.869281,0.862745,0.850770
1,logistic,recording,102,100,2,0.980392,0.979401,0.983146,0.979026,0.978758,0.980392,0.976471
2,baseline,beat,2588,1932,656,0.746522,0.763242,0.765011,0.745317,0.773984,0.746522,0.739974
3,logistic,beat,2588,2215,373,0.855873,0.861284,0.847250,0.842311,0.863765,0.855873,0.848094


{'train_recordings': 206, 'test_recordings': 102, 'train_beats': 5148, 'test_beats': 2588, 'all_beats': 7753, 'classes': 89, 'excluded_recordings': 2, 'iterations': [147], 'source_files_unchanged': True}


## טבלת הפעימות ובדיקת החלוקה
הטבלה המלאה נמצאת ב־experiment['beats'] וב־beats.csv.gz. כדי לא להציג אותות ביומטריים ללא צורך, מוצגים כאן רק הממדים ושמות העמודות. beat_id הוא מיקום פסגת R בדגימות בהקלטה; recording_id כולל גם את מזהה האדם. הערכים T1…T350 מסוננים ומנורמלים, ואינם אות גולמי.

In [2]:
table = experiment['beats']
print('Beat table shape:', table.shape)
print('Columns:', list(table.columns[:5]), '...', list(table.columns[-3:]))
train, test = experiment['train'], experiment['test']
assert set(train.recording_id).isdisjoint(test.recording_id)
assert set(test.person_id) <= set(train.person_id)
print('Recording overlap:', len(set(train.recording_id) & set(test.recording_id)))
print('Test people absent from training:', len(set(test.person_id) - set(train.person_id)))
print('Scaler fit samples:', experiment['model'].named_steps['scaler'].n_samples_seen_)
print('Training beats:', len(train))

Beat table shape: (7753, 353)
Columns: ['person_id', 'recording_id', 'beat_id', 'T1', 'T2'] ... ['T348', 'T349', 'T350']
Recording overlap: 0
Test people absent from training: 0
Scaler fit samples: 5148.0
Training beats: 5148


## מטריצות בלבול, מדדים לכל אדם וטעויות בולטות
לכל רמה מוצגים אותם נתוני בדיקה לשתי השיטות. שורה במטריצה היא האדם האמיתי, ועמודה היא האדם החזוי. טבלת הטעויות סופרת כמה פעמים כל זוג שיוך שגוי הופיע. מדדי macro נותנים משקל שווה לאנשים; weighted משקללים לפי מספר דוגמאות הבדיקה.

In [3]:
import plotly.graph_objects as go
for (unit, method), (report, cm, errors) in experiment['evaluations'].items():
    print(unit, method)
    display(report)
    display(errors.head(15))
    pair_count = int(cm.loc['Person_52', 'Person_64'])
    print('Person_52 -> Person_64:', pair_count, unit)
    fig = go.Figure(go.Heatmap(z=cm.to_numpy(), x=cm.columns.tolist(), y=cm.index.tolist(), colorscale='Blues'))
    fig.update_layout(title=f'{unit} — {method}', xaxis_title='Predicted person', yaxis_title='True person', yaxis_autorange='reversed', height=650)
    display(fig)

recording baseline


,precision,recall,f1-score,support
Person_01,1.000000,0.750000,0.857143,4.000000
Person_02,1.000000,0.600000,0.750000,5.000000
Person_03,1.000000,1.000000,1.000000,1.000000
Person_04,1.000000,1.000000,1.000000,1.000000
Person_05,1.000000,1.000000,1.000000,1.000000
...,...,...,...,...
Person_89,1.000000,1.000000,1.000000,1.000000
Person_90,1.000000,1.000000,1.000000,1.000000
accuracy,0.862745,0.862745,0.862745,0.862745
macro avg,0.859551,0.889700,0.864125,102.000000


,true,predicted,count
7,Person_52,Person_64,2
0,Person_01,Person_30,1
1,Person_02,Person_20,1
2,Person_02,Person_42,1
3,Person_09,Person_51,1
4,Person_30,Person_69,1
5,Person_36,Person_49,1
6,Person_51,Person_09,1
8,Person_59,Person_51,1
9,Person_62,Person_76,1


Person_52 -> Person_64: 2 recording


recording logistic


,precision,recall,f1-score,support
Person_01,1.000000,1.000000,1.000000,4.000000
Person_02,1.000000,1.000000,1.000000,5.000000
Person_03,0.500000,1.000000,0.666667,1.000000
Person_04,1.000000,1.000000,1.000000,1.000000
Person_05,1.000000,1.000000,1.000000,1.000000
...,...,...,...,...
Person_89,1.000000,1.000000,1.000000,1.000000
Person_90,1.000000,1.000000,1.000000,1.000000
accuracy,0.980392,0.980392,0.980392,0.980392
macro avg,0.979401,0.983146,0.979026,102.000000


... (6 outputs left out)

beat logistic


,precision,recall,f1-score,support
Person_01,0.784000,1.000000,0.878924,98.000000
Person_02,0.856115,0.901515,0.878229,132.000000
Person_03,0.615385,1.000000,0.761905,24.000000
Person_04,0.680000,0.739130,0.708333,23.000000
Person_05,0.941176,0.941176,0.941176,17.000000
...,...,...,...,...
Person_89,1.000000,0.954545,0.976744,22.000000
Person_90,0.892857,0.961538,0.925926,26.000000
accuracy,0.855873,0.855873,0.855873,0.855873
macro avg,0.861284,0.847250,0.842311,2588.000000


,true,predicted,count
150,Person_62,Person_72,22
86,Person_32,Person_03,12
198,Person_87,Person_01,12
182,Person_80,Person_01,8
188,Person_85,Person_16,8
134,Person_52,Person_64,7
2,Person_02,Person_48,6
6,Person_04,Person_37,5
64,Person_20,Person_47,5
136,Person_55,Person_45,5


Person_52 -> Person_64: 7 beat


## פירוש התוצאה
המספר המקורי 88/102 הוא ברמת הקלטה. יש להשוות אותו לתוצאת Logistic Regression ברמת recording בלבד. ברמת beat יש מבחן נוסף של זיהוי פעימה יחידה. התא הבא מפיק את המסקנה מההרצה בפועל, בלי מספרים קבועים מראש.
ההפרדה לפי הקלטות מונעת מעבר פעימות קשורות בין אימון לבדיקה. הסינון הדו־כיווני וסינון המשרעת משתמשים בהקלטה השלמה הזמינה לניבוי, ולכן אין כאן בדיקת זמן אמת. החלוקה אינה בהכרח בין תאריכים, והניסוי אינו מוכיח זיהוי תחת לחץ או זיהוי אדם לא מוכר.

In [4]:
scores = experiment['summary'].query("unit == 'recording'").set_index('method')
for method, row in scores.iterrows():
    print(f"{method}: {int(row.correct)}/{int(row.total)} = {row.accuracy:.2%}; errors={int(row.errors)}")
difference = scores.loc['logistic', 'accuracy'] - scores.loc['baseline', 'accuracy']
print(f'Logistic minus baseline: {100*difference:+.2f} percentage points')
print('Interactive local report:', project / 'derived_session3' / 'comparison.html')

baseline: 88/102 = 86.27%; errors=14
logistic: 100/102 = 98.04%; errors=2
Logistic minus baseline: +11.76 percentage points
Interactive local report: C:\Users\henb4\Desktop\התיקייה של חן\קריירה\פרויקט דאטה סיינס\דאטה בייס\ecg-id-database-1.0.0\derived_session3\comparison.html
